In [0]:
AWS_ACCESS_KEY = dbutils.secrets.get(scope="aws-credentials", key="access-key")
AWS_SECRET_KEY = dbutils.secrets.get(scope="aws-credentials", key="secret-key")
BUCKET_NAME    = "enem-data-lake-gblzera"
REGION         = "sa-east-1"

print("Credenciais carregadas com sucesso")

In [0]:
import boto3

s3 = boto3.client(
    "s3",
    aws_access_key_id=AWS_ACCESS_KEY,
    aws_secret_access_key=AWS_SECRET_KEY,
    region_name=REGION
)

buckets = s3.list_buckets()
for b in buckets["Buckets"]:
    print(f"Bucket encontrado: {b['Name']}")

In [0]:
import requests
import zipfile
import io
import os
from tqdm import tqdm

ENEM_URLS = {
    2022: "https://download.inep.gov.br/microdados/microdados_enem_2022.zip",
    2021: "https://download.inep.gov.br/microdados/microdados_enem_2021.zip",
}

def download_and_upload_to_s3(year: int, url: str):
    print(f"\n[{year}] Iniciando download...")

    response = requests.get(url, stream=True, timeout=600)
    response.raise_for_status()

    total_size = int(response.headers.get("content-length", 0))
    print(f"[{year}] Tamanho: {total_size / 1e9:.2f} GB")

    zip_content = io.BytesIO()
    for chunk in tqdm(response.iter_content(chunk_size=8192),
                      total=total_size//8192,
                      desc=f"Download {year}"):
        zip_content.write(chunk)

    zip_content.seek(0)
    print(f"[{year}] Extraindo e enviando para S3...")

    with zipfile.ZipFile(zip_content) as z:
        for file_name in z.namelist():
            if file_name.endswith(".csv") or file_name.endswith(".CSV"):
                print(f"  → Enviando: {file_name}")
                with z.open(file_name) as f:
                    s3_key = f"bronze/enem/{year}/{os.path.basename(file_name)}"
                    s3.upload_fileobj(f, BUCKET_NAME, s3_key)
                    print(f"  ✓ s3://{BUCKET_NAME}/{s3_key}")

# testar o primeiro ano
download_and_upload_to_s3(2022, ENEM_URLS[2022])